In [ ]:
# 1. Cài đặt thư viện cần thiết
!pip install pyspark gcsfs -q

In [2]:
# 2. Xác thực tài khoản Google
from google.colab import auth
auth.authenticate_user()

# 3. Khởi tạo Spark với các cấu hình tối ưu
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("AmazonReviews_EndToEnd") \
    .config("spark.jars.packages", "com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.5") \
    .config("spark.sql.caseSensitive", "true") \
    .getOrCreate()

print("Khởi tạo Spark thành công!")

Khởi tạo Spark thành công!


In [ ]:
import gcsfs
import pandas as pd
import concurrent.futures
from google.colab import auth
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
# Khởi tạo FileSystem kết nối với Google Cloud
fs = gcsfs.GCSFileSystem()

bucket_path = "amazon-reviews-lakehouse-data/bronze-zone"

# Quét danh sách các thư mục con 
meta_folders = [path.split('/')[-1] for path in fs.ls(f"{bucket_path}/meta-data/") if not path.endswith('_SUCCESS')]
review_folders = [path.split('/')[-1] for path in fs.ls(f"{bucket_path}/review-data/") if not path.endswith('_SUCCESS')]

print("--- THỐNG KÊ DANH MỤC (CATEGORIES) ---")
print(f"Tổng số Category trong Meta-data: {len(meta_folders)}")
print(f"Tổng số Category trong Review-data: {len(review_folders)}")

# Tìm xem có Category nào bị lệch giữa Meta và Review không
missing_in_review = set(meta_folders) - set(review_folders)
if missing_in_review:
    print(f"Các Category có trong Meta nhưng thiếu trong Review: {missing_in_review}")
else:
    print("Dữ liệu Meta và Review khớp nhau hoàn toàn về số lượng danh mục!")

--- THỐNG KÊ DANH MỤC (CATEGORIES) ---
Tổng số Category trong Meta-data: 32
Tổng số Category trong Review-data: 32
Dữ liệu Meta và Review khớp nhau hoàn toàn về số lượng danh mục!


In [ ]:
if 'spark' in locals():
    spark.stop()

from google.colab import auth
auth.authenticate_user()

# Khởi tạo lại Spark với ĐẦY ĐỦ cấu hình GCS Connector
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviews_Profiling") \
    .config("spark.jars.packages", "com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.5") \
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS") \
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .getOrCreate()

print("Đã khởi tạo lại Spark với GCS Connector thành công! Đường ống gs:// đã mở.")

Đã khởi tạo lại Spark với GCS Connector thành công! Đường ống gs:// đã mở.


In [ ]:
import math
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

# --- 1. RESET VÀ CẤU HÌNH LẠI RAM ---
if 'spark' in locals():
    spark.stop()

spark = SparkSession.builder \
    .appName("AmazonReviews_Profiling_Batches") \
    .config("spark.jars.packages", "com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.5") \
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS") \
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .config("spark.driver.memory", "6g") \
    .config("spark.executor.memory", "6g") \
    .getOrCreate()
print("Đã khởi tạo lại Spark với cấu hình an toàn!")

# --- 2. HÀM PHÂN TÍCH ĐƠN LẺ ---
def analyze_single_category(cat, base_gs_path, data_type_label):
    if not cat: return None

    path = f"gs://{base_gs_path}/{cat}/*.parquet"
    summary_list = []

    try:
        df = spark.read.option("mergeSchema", "true").parquet(path)
        total_rows = df.count()
        dtypes_dict = dict(df.dtypes)

        # Xây dựng biểu thức đếm Null
        null_exprs = [
            F.sum(F.when(F.col(c).isNull() | F.isnan(F.col(c)), 1).otherwise(0)).alias(c)
            if t in ["double", "float"] else
            F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
            for c, t in df.dtypes
        ]

        null_counts = df.agg(*null_exprs).collect()[0].asDict()

        for c in df.columns:
            summary_list.append({
                "Vùng_Dữ_Liệu": data_type_label,
                "Danh_Mục": cat,
                "Tên_Cột": c,
                "Kiểu_Dữ_Liệu": dtypes_dict[c],
                "Tổng_Số_Dòng": total_rows,
                "Số_Dòng_Null": null_counts[c],
                "Tỷ_Lệ_Null_(%)": round((null_counts[c] / total_rows) * 100, 2) if total_rows > 0 else 0
            })
        print(f" Xong: {cat} ({total_rows:,} dòng)")
        return summary_list

    except Exception as e:
        print(f" Lỗi ở {cat}: {str(e).split(chr(10))[0]}")
        return None

# --- 3. HÀM CHIA BATCH ---
def profile_datalake_batches(base_gs_path, categories, data_type_label, num_batches=4):
    print(f"\n Bắt đầu quét {data_type_label} theo {num_batches} batches...")

    # Chia đều mảng danh mục thành các chunks
    chunk_size = math.ceil(len(categories) / num_batches)
    chunks = [categories[i:i + chunk_size] for i in range(0, len(categories), chunk_size)]

    all_dfs = []

    for i, chunk in enumerate(chunks):
        batch_num = i + 1
        print(f"\n---> Đang xử lý Batch {batch_num}/{len(chunks)} ({len(chunk)} danh mục) <---")

        batch_results = []
        for cat in chunk:
            # Chạy tuần tự, để Spark tự phân tán bên dưới
            result = analyze_single_category(cat, base_gs_path, data_type_label)
            if result:
                batch_results.extend(result)

        # Xử lý và lưu tạm kết quả sau mỗi batch
        if batch_results:
            df_batch = pd.DataFrame(batch_results)
            all_dfs.append(df_batch)

            # Lưu file CSV dự phòng
            checkpoint_file = f"checkpoint_{data_type_label.lower()}_batch_{batch_num}.csv"
            df_batch.to_csv(checkpoint_file, index=False)
            print(f"Đã lưu tạm checkpoint vào: {checkpoint_file}")

    # Gộp toàn bộ kết quả
    if all_dfs:
        df_final = pd.concat(all_dfs, ignore_index=True)
        print(f"\n Hoàn thành gộp {len(all_dfs)} batches cho {data_type_label}!")
        return df_final
    return pd.DataFrame()

# --- 4. THỰC THI CHIA BATCH ---
# Thay đổi num_batches (3, 4, hoặc 5) tùy theo nhu cầu
print("Đang thống kê Review...")
df_report_review = profile_datalake_batches(f"{bucket_path}/review-data", review_folders, "Review", num_batches=4)

print("\nĐang thống kê Meta...")
df_report_meta = profile_datalake_batches(f"{bucket_path}/meta-data", meta_folders, "Meta", num_batches=4)

# Gộp bảng cuối cùng
df_final_report = pd.concat([df_report_review, df_report_meta], ignore_index=True)

print("\n HOÀN THÀNH TỔNG HỢP TOÀN BỘ!")
display(df_final_report)

✅ Đã khởi tạo lại Spark với cấu hình an toàn!
Đang thống kê Review...

🚀 Bắt đầu quét Review theo 4 batches...

---> Đang xử lý Batch 1/4 (8 danh mục) <---
  ✅ Xong: All_Beauty (701,528 dòng)
  ✅ Xong: Amazon_Fashion (2,500,939 dòng)
  ✅ Xong: Appliances (2,128,605 dòng)
  ✅ Xong: Arts_Crafts_and_Sewing (8,966,758 dòng)
  ✅ Xong: Automotive (19,955,450 dòng)
  ✅ Xong: Books (29,475,453 dòng)
  ✅ Xong: CDs_and_Vinyl (4,827,273 dòng)
💾 Đã lưu tạm checkpoint vào: checkpoint_review_batch_1.csv

---> Đang xử lý Batch 2/4 (8 danh mục) <---
  ✅ Xong: Cell_Phones_and_Accessories (20,812,945 dòng)
  ✅ Xong: Clothing_Shoes_and_Jewelry (66,033,346 dòng)


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: reentrant call inside <_io.BufferedReader name=47>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 566, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in sen

  ❌ Lỗi ở Digital_Music: An error occurred while calling o1187.count


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: reentrant call inside <_io.BufferedReader name=47>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 566, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in sen

  ❌ Lỗi ở Electronics: An error occurred while calling o1193.count
  ❌ Lỗi ở Gift_Cards: [Errno 111] Connection refused
  ❌ Lỗi ở Grocery_and_Gourmet_Food: [Errno 111] Connection refused
  ❌ Lỗi ở Health_and_Household: [Errno 111] Connection refused
  ❌ Lỗi ở Health_and_Personal_Care: [Errno 111] Connection refused
💾 Đã lưu tạm checkpoint vào: checkpoint_review_batch_2.csv

---> Đang xử lý Batch 3/4 (8 danh mục) <---
  ❌ Lỗi ở Home_and_Kitchen: [Errno 111] Connection refused
  ❌ Lỗi ở Industrial_and_Scientific: [Errno 111] Connection refused
  ❌ Lỗi ở Kindle_Store: [Errno 111] Connection refused
  ❌ Lỗi ở Magazine_Subscriptions: [Errno 111] Connection refused
  ❌ Lỗi ở Movies_and_TV: [Errno 111] Connection refused
  ❌ Lỗi ở Musical_Instruments: [Errno 111] Connection refused
  ❌ Lỗi ở Office_Products: [Errno 111] Connection refused
  ❌ Lỗi ở Patio_Lawn_and_Garden: [Errno 111] Connection refused

---> Đang xử lý Batch 4/4 (8 danh mục) <---
  ❌ Lỗi ở Pet_Supplies: [Errno 111] Connection 

,Vùng_Dữ_Liệu,Danh_Mục,Tên_Cột,Kiểu_Dữ_Liệu,Tổng_Số_Dòng,Số_Dòng_Null,Tỷ_Lệ_Null_(%)
0,Review,All_Beauty,asin,string,701528,0,0.0
1,Review,All_Beauty,helpful_vote,bigint,701528,0,0.0
2,Review,All_Beauty,images,"array<struct<attachment_type:string,large_imag...",701528,0,0.0
3,Review,All_Beauty,parent_asin,string,701528,0,0.0
4,Review,All_Beauty,rating,double,701528,0,0.0
...,...,...,...,...,...,...,...
85,Review,Clothing_Shoes_and_Jewelry,text,string,66033346,0,0.0
86,Review,Clothing_Shoes_and_Jewelry,timestamp,bigint,66033346,0,0.0
87,Review,Clothing_Shoes_and_Jewelry,title,string,66033346,0,0.0
88,Review,Clothing_Shoes_and_Jewelry,user_id,string,66033346,0,0.0


Tiền xử lý dữ liệu


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def process_review_silver(df):
    # 1. Thêm cột audit: source_category
    df = df.withColumn("source_category", F.element_at(F.split(F.input_file_name(), "/"), -2))

    # 2. Xử lý Missing Value: Drop record thiếu các khóa quan trọng
    df = df.dropna(subset=["user_id", "parent_asin", "rating"])

    # 3. Chuẩn hóa kiểu dữ liệu & Ép kiểu
    df = df.withColumnRenamed("timestamp", "timestamp_raw") \
           .withColumn("timestamp", F.to_timestamp(F.col("timestamp_raw") / 1000)) \
           .withColumn("rating", F.col("rating").cast("byte")) \
           .withColumn("verified_purchase", F.col("verified_purchase").cast("boolean")) \
           .withColumn("helpful_vote", F.col("helpful_vote").cast("int"))

    # 4. Chuẩn hóa Text (Bỏ HTML, URL, ký tự đặc biệt, trim )
    df = df.withColumn("text", F.coalesce(F.col("text"), F.lit(""))) \
           .withColumn("text", F.regexp_replace(F.col("text"), r"<[^>]*>", " ")) \
           .withColumn("text", F.regexp_replace(F.col("text"), r"https?://\S+|www\.\S+", "")) \
           .withColumn("text", F.regexp_replace(F.col("text"), r"[^a-zA-Z0-9\s]", "")) \
           .withColumn("text", F.trim(F.regexp_replace(F.col("text"), r"\s+", " ")))

    # 5. Xử lý trùng lặp (Deduplicate) theo khóa
    window_dedup = Window.partitionBy("user_id", "parent_asin", "timestamp_raw").orderBy(F.col("helpful_vote").desc_nulls_last())
    df = df.withColumn("rn", F.row_number().over(window_dedup)).filter(F.col("rn") == 1).drop("rn")

    # 6. Thêm cột audit: processed_at
    df = df.withColumn("processed_at", F.current_timestamp())

    # 7. Giữ lại các cột lõi
    core_columns = ["user_id", "parent_asin", "rating", "text", "timestamp", "timestamp_raw", "verified_purchase", "helpful_vote", "source_category", "processed_at"]
    df = df.select(*[c for c in core_columns if c in df.columns])

    return df

In [9]:
def process_meta_silver(df):
    # 1. Thêm cột audit: source_category
    df = df.withColumn("source_category", F.element_at(F.split(F.input_file_name(), "/"), -2))

    # 2. Xử lý Missing Value: Drop record thiếu parent_asin
    df = df.dropna(subset=["parent_asin"])

    # 3. Chuẩn hóa kiểu dữ liệu & Ép kiểu
    df = df.withColumn("average_rating", F.col("average_rating").cast("float")) \
           .withColumn("rating_number", F.col("rating_number").cast("int"))

    # Parse price (lấy số từ chuỗi)
    if "price" in df.columns:
        df = df.withColumn("price", F.regexp_extract(F.col("price"), r"(\d+\.\d+|\d+)", 1).cast("float"))

    # 4. Chuẩn hóa Text đơn giản (title, store)
    if "title" in df.columns:
        df = df.withColumn("title", F.trim(F.coalesce(F.col("title"), F.lit(""))))
    if "store" in df.columns:
        df = df.withColumn("store", F.trim(F.coalesce(F.col("store"), F.lit(""))))

    # 5. Chuẩn hóa Mảng (Tránh null gây lỗi logic mảng)
    array_cols = ["categories", "features", "description"]
    for c in array_cols:
        if c in df.columns:
            df = df.withColumn(c, F.when(F.col(c).isNull(), F.array()).otherwise(F.col(c)))

    # 6. Deduplicate theo parent_asin (Ưu tiên bản ghi có info đầy đủ nhất)
    score_expr = F.lit(0)
    if "title" in df.columns: score_expr += F.when(F.length(F.col("title")) > 0, 1).otherwise(0)
    if "store" in df.columns: score_expr += F.when(F.length(F.col("store")) > 0, 1).otherwise(0)
    if "price" in df.columns: score_expr += F.when(F.col("price").isNotNull(), 1).otherwise(0)
    if "average_rating" in df.columns: score_expr += F.when(F.col("average_rating").isNotNull(), 1).otherwise(0)

    df = df.withColumn("info_score", score_expr)
    window_meta = Window.partitionBy("parent_asin").orderBy(F.col("info_score").desc_nulls_last())
    df = df.withColumn("rn", F.row_number().over(window_meta)).filter(F.col("rn") == 1).drop("rn", "info_score")

    # 7. Thêm cột audit: processed_at
    df = df.withColumn("processed_at", F.current_timestamp())

    return df

Đẩy data lên silver-zone


In [ ]:
# --- CELL 1: ĐỊNH NGHĨA HÀM ETL CÓ CƠ CHẾ KIỂM TRA TỒN TẠI ---
import gcsfs

# Khởi tạo filesystem để quét file trên GCS
fs = gcsfs.GCSFileSystem()

bronze_base = "amazon-reviews-lakehouse-data/bronze-zone"
silver_base = "amazon-reviews-lakehouse-data/silver-zone"

def execute_silver_pipeline(categories, sub_folder, data_type, skip_if_exists=True):
    print(f"\n🚀 Bắt đầu tiến trình ETL cho nhánh: {sub_folder} (Tổng: {len(categories)} danh mục)")

    success_count = 0
    skip_count = 0
    failed_categories = []

    for cat in categories:
        source_path = f"gs://{bronze_base}/{sub_folder}/{cat}/*.parquet"
        target_path = f"gs://{silver_base}/{sub_folder}/{cat}"

        # --- KIỂM TRA NẾU ĐÃ TỒN TẠI THÌ BỎ QUA ---
        if skip_if_exists:
            # Nếu đã có file _SUCCESS nghĩa là Spark đã ghi thành công ở lần chạy trước
            if fs.exists(f"{target_path}/_SUCCESS"):
                print(f"⏩ Bỏ qua: {cat} (Đã có ở Silver)")
                skip_count += 1
                continue

        print(f"⏳ Đang xử lý: {cat}...")

        try:
            # 1. Đọc dữ liệu từ Bronze Zone
            df_bronze = spark.read.option("mergeSchema", "true").parquet(source_path)

            # 2. Gọi hàm tiền xử lý tương ứng
            if data_type == "Review":
                df_silver = process_review_silver(df_bronze)
            elif data_type == "Meta":
                df_silver = process_meta_silver(df_bronze)

            # 3. Ghi dữ liệu xuống Silver Zone (Gom thành 1 file duy nhất)
            df_silver.coalesce(1).write \
                .mode("overwrite") \
                .parquet(target_path)

            print(f" Đã ghi thành công: {cat}")
            success_count += 1

        except Exception as e:
            print(f" LỖI tại {cat}: {str(e).split(chr(10))[0]}")
            failed_categories.append(cat)

    print(f"\n HOÀN THÀNH NHÁNH {data_type.upper()}!")
    print(f" Thành công: {success_count} | ⏩ Bỏ qua: {skip_count} | ⚠️ Lỗi: {len(failed_categories)}")

In [12]:
# --- CELL 2: CHẠY THỬ NGHIỆM VỚI DANH MỤC NHỎ ---
test_category = ["Gift_Cards"]

print("--- TEST ĐẨY NHÁNH META-DATA ---")
execute_silver_pipeline(
    categories=test_category,
    sub_folder="meta-data",
    data_type="Meta"
)

print("\n--- TEST ĐẨY NHÁNH REVIEW-DATA ---")
execute_silver_pipeline(
    categories=test_category,
    sub_folder="review-data",
    data_type="Review"
)

--- TEST ĐẨY NHÁNH META-DATA ---

🚀 Bắt đầu tiến trình ETL cho nhánh: meta-data (Tổng: 1 danh mục)
⏩ Bỏ qua: Gift_Cards (Đã có ở Silver)

🎉 HOÀN THÀNH NHÁNH META!
✅ Thành công: 0 | ⏩ Bỏ qua: 1 | ⚠️ Lỗi: 0

--- TEST ĐẨY NHÁNH REVIEW-DATA ---

🚀 Bắt đầu tiến trình ETL cho nhánh: review-data (Tổng: 1 danh mục)
⏩ Bỏ qua: Gift_Cards (Đã có ở Silver)

🎉 HOÀN THÀNH NHÁNH REVIEW!
✅ Thành công: 0 | ⏩ Bỏ qua: 1 | ⚠️ Lỗi: 0


In [8]:
# --- CELL 3: KIỂM TRA KẾT QUẢ TRÊN GOOGLE CLOUD STORAGE ---
print("Danh sách file trong thư mục Meta (Silver):")
!gsutil ls -lh gs://amazon-reviews-lakehouse-data/silver-zone/meta-data/Gift_Cards/

print("\nDanh sách file trong thư mục Review (Silver):")
!gsutil ls -lh gs://amazon-reviews-lakehouse-data/silver-zone/review-data/Gift_Cards/

Danh sách file trong thư mục Meta (Silver):
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
       0 B  2026-03-30T03:50:45Z  gs://amazon-reviews-lakehouse-data/silver-zone/meta-data/Gift_Cards/
       0 B  2026-03-30T03:50:47Z  gs://amazon-reviews-lakehouse-data/silver-zone/meta-data/Gift_Cards/_SUCCESS
365.14 KiB  2026-03-30T03:50:42Z  gs://amazon-reviews-lakehouse-data/silver-zone/meta-data/Gift_Cards/part-00000-31cff9a4-221e-4c5d-a632-daf3109b3550-c000.snappy.parquet
TOTAL: 3 objects, 373901 bytes (365.14 KiB)

Danh sách file trong thư mục Review (Silver):
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-t

In [ ]:
# --- CELL 4: CHẠY CHÍNH THỨC CHO TOÀN BỘ DỮ LIỆU ---
# Lưu ý: Biến meta_folders và review_folders là mảng danh sách các folder
# mà bạn đã khai báo ở đầu Notebook.

print("--- 1. CHẠY TOÀN BỘ NHÁNH META-DATA ---")
execute_silver_pipeline(
    categories=meta_folders,
    sub_folder="meta-data",
    data_type="Meta"
)

print("\n--- 2. CHẠY TOÀN BỘ NHÁNH REVIEW-DATA ---")
execute_silver_pipeline(
    categories=review_folders,
    sub_folder="review-data",
    data_type="Review"
)